# Business Card Lead Extractor — Colab backend
Run the project's actual FastAPI backend and pretrained **Qwen2.5-VL-3B-Instruct** on a Colab NVIDIA GPU. No model training is involved.

**Keep your existing `business-card-lead-extractor-source.zip` unchanged.** Upload this notebook through **File → Upload notebook**. Select **Runtime → Change runtime type → T4 GPU**. Run numbered sections in order, waiting for each cell to finish. Do not use a local runtime.

The notebook locates your original ZIP in `/content` (or offers an upload picker), extracts a new working copy, and applies a small documented Colab compatibility patch automatically. Only the notebook and original ZIP are required; test cards are generated on Colab.

The backend listens on Colab's own `127.0.0.1:8000`; requests come from notebook cells. This is interactive backend testing, not public hosting or a connection to your laptop's React preview. Docker, Nginx and React are not started. [Colab usage restrictions and resource limits](https://research.google.com/colaboratory/faq.html).

**Validation status:** authored and statically reviewed; GPU execution and test results must be established by running this notebook. Do not describe unexecuted cells as passing. Model license: [Qwen Research License](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/blob/main/LICENSE).

## 1. Confirm the Colab runtime and GPU
This cell does not load a model. A T4 normally reports about 15 GiB of usable VRAM. If no NVIDIA GPU is available, change the runtime type or retry later; there is no CPU/TPU fallback. Disk thresholds are planning allowances, not a guarantee that every workload fits.

In [ ]:
import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path

try:
    from google.colab import files as colab_files
except ImportError as exc:
    raise RuntimeError("Open this notebook in hosted Google Colab, not on your laptop.") from exc

def require_colab():
    if sys.platform != "linux" or not Path("/content").is_dir():
        raise RuntimeError("Use a hosted Colab GPU runtime; do not run this notebook locally.")

require_colab()
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("This notebook targets Python 3.11–3.13. Check dependency compatibility before proceeding.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No NVIDIA GPU found. Select Runtime → Change runtime type → T4 GPU.")
gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
     "--format=csv,noheader"], text=True
).strip()
print("GPU: name, total MiB, free MiB, driver\n" + gpu_info)
gpu_free_mib = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
    text=True
).splitlines()[0])
if gpu_free_mib < 13000:
    raise RuntimeError("Less than 13,000 MiB GPU memory is free. Release other GPU work or use a fresh T4 runtime.")
disk_free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"Python: {sys.version.split()[0]} | free disk: {disk_free_gib:.1f} GiB")
if disk_free_gib < 30:
    raise RuntimeError("Reserve at least 30 GiB free disk for the isolated environment and model cache.")
print("Runtime checks passed. No model has been loaded.")


## 2. Locate your original ZIP and extract a separate working copy
The default path matches the upload already described. If Colab renamed a duplicate upload, change `ZIP_PATH` below to its exact path, for example `/content/business-card-lead-extractor-source (1).zip`. Do not upload keys or an environment file.

Re-running this cell creates a fresh source copy. Stop the backend first using section 12 if it is already running. The ZIP and any earlier manual extraction are left intact.

In [ ]:
require_colab()
import hashlib
import stat
import tempfile
import zipfile

if globals().get("backend_process") is not None and backend_process.poll() is None:
    raise RuntimeError("Stop the backend in section 12 before extracting another source copy.")

ZIP_PATH = Path("/content/business-card-lead-extractor-source.zip")
if not ZIP_PATH.is_file():
    print("Select your unchanged business-card-lead-extractor-source.zip.")
    uploaded_zip = colab_files.upload()
    candidates = [Path(name).resolve() for name in uploaded_zip if name.lower().endswith(".zip")]
    del uploaded_zip
    if len(candidates) != 1:
        raise RuntimeError("Upload exactly one source ZIP, then rerun this cell.")
    ZIP_PATH = candidates[0]

archive_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
RUN_ROOT = Path(tempfile.mkdtemp(prefix="business-card-colab-", dir="/content"))
with zipfile.ZipFile(ZIP_PATH) as archive:
    entries = archive.infolist()
    if len(entries) > 2000 or sum(entry.file_size for entry in entries) > 100 * 1024**2:
        raise RuntimeError("Use the small source-only ZIP, without dependencies or model weights.")
    for entry in entries:
        destination = (RUN_ROOT / entry.filename).resolve()
        if not destination.is_relative_to(RUN_ROOT.resolve()):
            raise RuntimeError("The ZIP contains a path outside its extraction directory.")
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
    archive.extractall(RUN_ROOT)

roots = [path.parent.parent for path in RUN_ROOT.rglob("backend/requirements.txt")]
if len(roots) != 1:
    raise RuntimeError("Expected exactly one backend/requirements.txt in the supplied ZIP.")
PROJECT = roots[0]
BACKEND = PROJECT / "backend"
REPORTS = RUN_ROOT / "reports"
REPORTS.mkdir()
for relative in [
    "app/main.py", "app/core/config.py", "app/services/vlm_service.py",
    "requirements-model.txt", "requirements-test.txt", "pyproject.toml",
    "scripts/generate_samples.py", "scripts/aws_smoke.py", "tests/test_api.py",
]:
    if not (BACKEND / relative).is_file():
        raise RuntimeError(f"Required source file is missing: backend/{relative}")

print("Project:", PROJECT)
print("Original ZIP SHA256:", archive_sha256)
print("The original ZIP has not been changed.")


## 3. Apply Colab compatibility changes automatically
The patch:
- Adds the explicit `colab` execution target alongside `aws` in Settings and the model-loading guard.
- Keeps inference disabled by default and still requires CUDA.
- Gives the two reused evaluation scripts a `--colab-only` flag and Colab wording.

The seven fields, Qwen model implementation, prompt, routes, image handling and Excel generation are unchanged. The patch stops if the expected original text does not match; it does not silently rewrite unknown source. Original file copies and the patch manifest are saved under this run's reports directory. The standalone AWS PoC is not used, avoiding a second model instance.

In [ ]:
require_colab()
if globals().get("backend_process") is not None and backend_process.poll() is None:
    raise RuntimeError("Stop the backend before changing its source.")

PATCHES = {
    "app/core/config.py": [
        ('Literal["disabled", "aws"]', 'Literal["disabled", "aws", "colab"]'),
        ("def require_explicit_aws_target(self):", "def require_explicit_cloud_target(self):"),
        ('self.execution_target != "aws"', 'self.execution_target not in {"aws", "colab"}'),
        ("Inference is allowed only with EXECUTION_TARGET=aws",
         "Inference is allowed only with EXECUTION_TARGET=aws or colab"),
    ],
    "app/services/vlm_service.py": [
        ('self.settings.execution_target != "aws"',
         'self.settings.execution_target not in {"aws", "colab"}'),
        ("Inference is permitted only on AWS.", "Inference is permitted only on AWS or Colab."),
        ("Check the AWS NVIDIA driver and container toolkit.",
         "Use a CUDA GPU on Colab, or check the AWS NVIDIA driver and container toolkit."),
    ],
    "scripts/generate_samples.py": [
        ("Generate fictional evaluation images on the AWS server only.",
         "Generate fictional evaluation images on the Colab runtime only."),
        ('"--aws-only"', '"--colab-only"'),
    ],
    "scripts/aws_smoke.py": [
        ("Run after deployment ON AWS.", "Run after backend startup ON COLAB."),
        ('"--aws-only"', '"--colab-only"'),
        ("AWS smoke checks passed.", "Colab backend smoke checks passed."),
    ],
}
MARKER = "# Colab compatibility patch v1: applied by business_card_backend.ipynb\n"
planned = []
for relative, replacements in PATCHES.items():
    path = BACKEND / relative
    source = path.read_text(encoding="utf-8")
    if source.startswith(MARKER):
        print("Already patched:", relative)
        continue
    updated = source
    for old, new in replacements:
        if updated.count(old) != 1:
            raise RuntimeError(f"Unexpected source in {relative}: expected exactly one {old!r}. Use the original ZIP.")
        updated = updated.replace(old, new, 1)
    planned.append((relative, path, source, MARKER + updated))

# Validate all replacements before writing any source file.
manifest_path = REPORTS / "colab-patch-manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else []
for relative, path, before, after in planned:
    backup = REPORTS / "original-source" / relative
    backup.parent.mkdir(parents=True, exist_ok=True)
    backup.write_text(before, encoding="utf-8")
    path.write_text(after, encoding="utf-8")
    manifest.append({
        "file": "backend/" + relative,
        "before_sha256": hashlib.sha256(before.encode()).hexdigest(),
        "after_sha256": hashlib.sha256(after.encode()).hexdigest(),
    })
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Colab patch ready. Inference is still disabled until section 6.")


## 4. Install dependencies into a separate Python environment
This downloads several gigabytes **onto Colab**. It avoids replacing the notebook's preinstalled Python packages. Wait for completion; if installation fails, stop and inspect that cell's error.

The PyTorch/CUDA pair matches the project's Dockerfile: [official PyTorch version pairs](https://pytorch.org/get-started/previous-versions/). The notebook uses Colab's Python interpreter, so this does not validate the Docker image's Python 3.11 environment.

If standard `venv` cannot bootstrap pip, this cell prints the underlying error and repairs the designated environment with the [official PyPA virtualenv zipapp](https://virtualenv.pypa.io/en/20.27.2/installation.html#via-zipapp). An existing Python executable alone is not treated as successful setup; isolation and pip must pass a check. No notebook runtime packages are replaced.


In [ ]:
require_colab()
if globals().get("backend_process") is not None and backend_process.poll() is None:
    raise RuntimeError("Stop the backend in section 12 before reinstalling its environment.")

VENV = Path("/content/business-card-backend-venv")
PYTHON = VENV / "bin/python"
# Any repair below is confined to the notebook's designated environment.
if VENV.is_symlink() or VENV.resolve() != Path("/content/business-card-backend-venv"):
    raise RuntimeError("Unexpected environment path. Refusing to repair a redirected directory.")

bootstrap_env = os.environ.copy()
bootstrap_env["PYTHONNOUSERSITE"] = "1"
bootstrap_env.pop("PYTHONPATH", None)

def environment_ready():
    if not PYTHON.is_file():
        return False
    probe = subprocess.run(
        [str(PYTHON), "-I", "-c",
         "import sys, pip; assert sys.prefix != sys.base_prefix; "
         f"assert sys.version_info[:2] == {sys.version_info[:2]!r}"],
        env=bootstrap_env, capture_output=True, text=True, timeout=30,
    )
    return probe.returncode == 0

if not environment_ready():
    creation = subprocess.run(
        [sys.executable, "-m", "venv", str(VENV)],
        env=bootstrap_env, capture_output=True, text=True, timeout=180,
    )
    (REPORTS / "venv-bootstrap.txt").write_text(
        creation.stdout + creation.stderr, encoding="utf-8"
    )
    if creation.returncode != 0 or not environment_ready():
        print("Standard venv did not create a usable environment:")
        print((creation.stdout + creation.stderr)[-6000:])
        print("Repairing with the official PyPA virtualenv zipapp; "
              "Colab's notebook packages will not be replaced.")
        from urllib.request import urlopen
        bootstrap_url = "https://bootstrap.pypa.io/virtualenv.pyz"
        bootstrap_path = RUN_ROOT / "virtualenv.pyz"
        with urlopen(bootstrap_url, timeout=60) as response:
            bootstrap_bytes = response.read()
        if not bootstrap_bytes:
            raise RuntimeError("The virtualenv bootstrap download was empty.")
        bootstrap_path.write_bytes(bootstrap_bytes)
        (REPORTS / "virtualenv-bootstrap.json").write_text(json.dumps({
            "url": bootstrap_url,
            "sha256": hashlib.sha256(bootstrap_bytes).hexdigest(),
        }, indent=2), encoding="utf-8")
        repair = subprocess.run(
            [sys.executable, str(bootstrap_path), "--clear", "--no-periodic-update",
             "--python", sys.executable, str(VENV)],
            env=bootstrap_env, capture_output=True, text=True, timeout=300,
        )
        (REPORTS / "virtualenv-repair.txt").write_text(
            repair.stdout + repair.stderr, encoding="utf-8"
        )
        print((repair.stdout + repair.stderr)[-6000:])
        repair.check_returncode()
    if not environment_ready():
        raise RuntimeError("Environment creation failed or pip is missing. "
                           "Inspect venv-bootstrap.txt and virtualenv-repair.txt in the reports.")
print("Isolated Python environment and pip are ready:", PYTHON)

def run_command(args, *, env=None, log_name=None, timeout=1800):
    require_colab()
    log_handle = (REPORTS / log_name).open("w", encoding="utf-8") if log_name else None
    try:
        result = subprocess.run(
            [str(arg) for arg in args], cwd=BACKEND, env=env,
            stdout=log_handle, stderr=subprocess.STDOUT if log_handle else None,
            timeout=timeout,
        )
    finally:
        if log_handle:
            log_handle.close()
    if log_name:
        print((REPORTS / log_name).read_text(encoding="utf-8", errors="replace")[-16000:])
    result.check_returncode()
    return result

install_env = os.environ.copy()
install_env["PYTHONNOUSERSITE"] = "1"
install_env.pop("PYTHONPATH", None)
run_command([PYTHON, "-m", "pip", "install", "--no-cache-dir",
             "--index-url", "https://pypi.org/simple", "--upgrade", "pip"], env=install_env)
run_command([PYTHON, "-m", "pip", "install", "--no-cache-dir",
             "torch==2.10.0", "torchvision==0.25.0",
             "--index-url", "https://download.pytorch.org/whl/cu126"], env=install_env)
run_command([PYTHON, "-m", "pip", "install", "--no-cache-dir",
             "--index-url", "https://pypi.org/simple",
             "-r", BACKEND / "requirements.txt",
             "-r", BACKEND / "requirements-model.txt",
             "-r", BACKEND / "requirements-test.txt"], env=install_env)
run_command([PYTHON, "-m", "pip", "check"], env=install_env, log_name="pip-check.txt")

# The existing synthetic card script needs this font; install it only if absent.
if not Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf").is_file():
    run_command(["apt-get", "update"], timeout=300)
    run_command(["apt-get", "install", "-y", "fonts-dejavu-core"], timeout=300)

# Version reporting is separate from dependency/runtime validation.
# Capture freeze's actual error instead of displaying only CalledProcessError.
freeze_result = subprocess.run(
    [str(PYTHON), "-m", "pip", "freeze"], cwd=BACKEND,
    text=True, env=install_env, capture_output=True, timeout=120,
)
(REPORTS / "package-inventory.json").unlink(missing_ok=True)
(REPORTS / "package-versions.txt").unlink(missing_ok=True)
if freeze_result.returncode == 0:
    (REPORTS / "pip-freeze.txt").write_text(freeze_result.stdout, encoding="utf-8")
    (REPORTS / "pip-freeze-error.txt").unlink(missing_ok=True)
    inventory_method = "pip freeze"
else:
    (REPORTS / "pip-freeze.txt").unlink(missing_ok=True)
    (REPORTS / "pip-freeze-error.txt").write_text(
        f"Exit code: {freeze_result.returncode}\n" +
        freeze_result.stdout + freeze_result.stderr, encoding="utf-8"
    )
    print("pip freeze failed during optional version reporting:")
    print((freeze_result.stdout + freeze_result.stderr)[-6000:])
    print("Recording installed versions using Python package metadata instead.")
    inventory_code = """
from importlib.metadata import distributions
rows = []
for package in distributions():
    name = package.metadata.get("Name")
    if name:
        rows.append(name + "==" + package.version)
print("\\n".join(sorted(set(rows), key=str.lower)))
"""
    metadata_result = subprocess.run(
        [str(PYTHON), "-I", "-c", inventory_code], cwd=BACKEND,
        text=True, env=install_env, capture_output=True, timeout=120,
    )
    if metadata_result.returncode != 0:
        print(metadata_result.stdout + metadata_result.stderr)
        metadata_result.check_returncode()
    (REPORTS / "package-versions.txt").write_text(
        metadata_result.stdout, encoding="utf-8"
    )
    inventory_method = "importlib.metadata name/version inventory; not a pip freeze lockfile"

(REPORTS / "package-inventory.json").write_text(json.dumps({
    "method": inventory_method,
    "pip_freeze_exit_code": freeze_result.returncode,
    "runtime_validation": "Continue with section 5; pip check alone does not prove imports or CUDA work.",
}, indent=2), encoding="utf-8")
print("Backend dependencies installed in", VENV)
print("Version inventory:", inventory_method)


## 5. Check the installed GPU stack and run non-model tests
These tests use the repository's explicitly injected test double. They check API behavior, parsing, validation and Excel handling; passing them does not prove real extraction works. Sections 7–10 use the actual Qwen model.

In [ ]:
require_colab()
test_env = install_env.copy()
test_env.update({
    "PYTHONPATH": str(BACKEND),
    "EXECUTION_TARGET": "disabled",
    "ENABLE_MODEL_INFERENCE": "false",
    "HF_HUB_DISABLE_TELEMETRY": "1",
})
run_command([PYTHON, "-c", """
import torch, torchvision, transformers
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
print("torch", torch.__version__, "torchvision", torchvision.__version__)
print("transformers", transformers.__version__, "CUDA wheel", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA is unavailable in the isolated backend environment"
print("GPU:", torch.cuda.get_device_name(0))
x = torch.ones(1, device="cuda")
assert x.item() == 1
print("CUDA allocation succeeded; no model loaded.")
"""], env=test_env, log_name="gpu-check.txt", timeout=180)

run_command([PYTHON, "-m", "pytest", "-p", "no:cacheprovider",
             "--junitxml=" + str(REPORTS / "pytest.xml")],
            env=test_env, log_name="pytest.txt", timeout=300)
print("Non-model tests passed. Real model validation is still pending.")


## 6. Configure the backend for Colab
This uses the supplied root environment example, including its exact model revision. FP16 is explicitly selected for T4. No AWS credentials or Hugging Face token are needed for the public checkpoint. The notebook does not read a private `.env`.

Defaults remain 20 cards, 10 MiB per card, 1,003,520 model pixels and 512 output tokens. Start with a single small card. Colab's available memory must still be confirmed under real inference.

In [ ]:
require_colab()
settings = {}
for line in (PROJECT / ".env.example").read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        settings[key.strip()] = value.strip()

EXPECTED_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
EXPECTED_REVISION = "66285546d2b821cf421d4f5eb2576359d3770cd3"
if settings.get("MODEL_ID") != EXPECTED_MODEL or settings.get("MODEL_REVISION") != EXPECTED_REVISION:
    raise RuntimeError("Unexpected model or revision in this ZIP. Review the source before running.")

CACHE = Path("/content/business-card-model-cache")
CACHE.mkdir(exist_ok=True)
TEMP = RUN_ROOT / "temporary"
TEMP.mkdir(exist_ok=True)
backend_env = install_env.copy()
backend_env.update(settings)
backend_env.update({
    "EXECUTION_TARGET": "colab",
    "ENABLE_MODEL_INFERENCE": "true",
    "MODEL_DTYPE": "float16",
    "CORS_ORIGINS": "[]",
    "ENABLE_API_DOCS": "false",
    "PYTHONPATH": str(BACKEND),
    "PYTHONUNBUFFERED": "1",
    "HF_HOME": str(CACHE),
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "TMPDIR": str(TEMP),
})
run_command([PYTHON, "-c", """
from app.core.config import Settings
s = Settings(_env_file=None)
assert s.execution_target == "colab" and s.enable_model_inference
print("Target:", s.execution_target, "| model:", s.model_id, "| dtype:", s.model_dtype)
print("Revision:", s.model_revision)
"""], env=backend_env, timeout=60)
(REPORTS / "run-settings.json").write_text(json.dumps({
    "execution_target": "colab", "model": EXPECTED_MODEL, "revision": EXPECTED_REVISION,
    "dtype": "float16", "archive_sha256": archive_sha256,
    "python": sys.version, "gpu": gpu_info,
    "limits": {key: backend_env[key] for key in (
        "MAX_FILES_PER_REQUEST", "MAX_UPLOAD_MB", "MODEL_MAX_PIXELS", "MODEL_MAX_NEW_TOKENS"
    )},
    "docker_nginx_react_validation": "not performed by this notebook",
}, indent=2), encoding="utf-8")
print("Settings checked; model download begins in the next section.")


## 7. Start one FastAPI process and wait for Qwen
The first run downloads pretrained weights and can take several minutes. Only one Uvicorn worker is started, without reload. The cell checks liveness and then waits up to 20 minutes for readiness. It prints progress every 15 seconds.

If a download continues after that wait, rerun this cell: it reuses the existing process instead of loading a second model. If the model fails, inspect section 11, stop with section 12, fix the cause, and rerun this section.

In [ ]:
require_colab()
import socket
from urllib.request import Request, build_opener, ProxyHandler
from urllib.error import HTTPError, URLError

BASE_URL = "http://127.0.0.1:8000"
LOG_PATH = REPORTS / "backend.log"
# Explicitly avoid routing loopback requests through an environment proxy.
http = build_opener(ProxyHandler({}))

def api_request(path, data=None, content_type=None, method=None, timeout=30):
    headers = {"Content-Type": content_type} if content_type else {}
    request = Request(BASE_URL + path, data=data, headers=headers, method=method)
    with http.open(request, timeout=timeout) as response:
        return response.status, dict(response.headers), response.read()

def api_json(path, **kwargs):
    return json.loads(api_request(path, **kwargs)[2])

def tail_backend(lines=60):
    if LOG_PATH.exists():
        print("\n".join(LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:]))

def stop_backend():
    process = globals().get("backend_process")
    if process is None or process.poll() is not None:
        print("No backend process owned by this notebook is running.")
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=15)
    print("Backend stopped. GPU model memory is released; cached weights stay on Colab disk.")

if globals().get("backend_process") is None or backend_process.poll() is not None:
    with socket.socket() as probe:
        try:
            probe.bind(("127.0.0.1", 8000))
        except OSError as exc:
            raise RuntimeError("Port 8000 is occupied by another process. Stop that process before proceeding.") from exc
    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n--- Starting Colab backend ---\n")
        log.flush()
        backend_process = subprocess.Popen(
            [str(PYTHON), "-m", "uvicorn", "app.main:app",
             "--host", "127.0.0.1", "--port", "8000", "--workers", "1",
             "--no-access-log", "--no-server-header"],
            cwd=BACKEND, env=backend_env, stdout=log, stderr=subprocess.STDOUT,
        )
    print("Started backend PID", backend_process.pid)
else:
    print("Reusing backend PID", backend_process.pid)

started_waiting = time.monotonic()
deadline = started_waiting + 1200
next_notice = 0
while True:
    if backend_process.poll() is not None:
        tail_backend()
        raise RuntimeError(f"Backend exited with code {backend_process.returncode}. Inspect the log above.")
    state = "starting API"
    try:
        state = api_json("/api/health", timeout=5)["model_state"]
        if state == "failed":
            tail_backend()
            raise RuntimeError("Model initialization failed. Inspect section 11, then stop the backend before retrying.")
        if state == "disabled":
            raise RuntimeError("Inference is disabled. Stop the backend, rerun section 6, and start it again.")
        if state == "ready":
            assert api_json("/api/ready", timeout=5) == {"ready": True}
            print("Qwen is ready. Backend:", BASE_URL, "(inside Colab only)")
            break
    except (URLError, TimeoutError, ConnectionError):
        pass
    elapsed = time.monotonic() - started_waiting
    if elapsed >= next_notice:
        print(f"{elapsed:.0f}s: {state}; waiting for model download/loading...")
        next_notice = elapsed + 15
    if time.monotonic() >= deadline:
        tail_backend()
        raise TimeoutError("Readiness wait expired. Inspect logs; rerun this cell to keep waiting, or section 12 to stop.")
    time.sleep(3)
print(json.dumps(api_json("/api/config"), indent=2))


## 8. Generate fictional cards and define the notebook API client
This reuses the project's sample generator. Input files and reports are stored on Colab's temporary disk. They remain there until you remove them or the runtime is deleted; this notebook does not reproduce Docker's tmpfs storage.

The displayed card is fictional. No user images are uploaded unless you explicitly choose them in section 10.

In [ ]:
require_colab()
from html import escape
from IPython.display import display, HTML, Image as DisplayImage
import uuid

SAMPLES = RUN_ROOT / "evaluation"
run_command([PYTHON, BACKEND / "scripts/generate_samples.py",
             "--colab-only", "--output", SAMPLES], env=backend_env, timeout=60)
display(DisplayImage(filename=str(SAMPLES / "standard.png"), width=600))

FIELDS = ("first_name", "last_name", "job_title", "company", "location", "phone", "email")

def display_results(job):
    rows = []
    for item in job["leads"]:
        lead = item.get("lead") or {}
        values = [item["source_filename"], item["status"]] + [
            lead.get(field) for field in FIELDS
        ] + [item.get("error") or "; ".join(item.get("warnings", []))]
        rows.append("<tr>" + "".join(
            "<td>" + escape("" if value is None else str(value)) + "</td>" for value in values
        ) + "</tr>")
    headers = ("file", "status") + FIELDS + ("error / warnings",)
    display(HTML(
        "<div style='overflow-x:auto'><table border='1' cellpadding='6'><thead><tr>" +
        "".join("<th>" + escape(field) + "</th>" for field in headers) +
        "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"
    ))

def submit_cards(paths):
    limits = api_json("/api/config")
    paths = [Path(path) for path in paths]
    if not 1 <= len(paths) <= limits["max_files"]:
        raise ValueError(f"Select between 1 and {limits['max_files']} cards.")
    boundary = "card-" + uuid.uuid4().hex
    payload = bytearray()
    for path in paths:
        if not path.is_file() or not 0 < path.stat().st_size <= limits["max_upload_mb"] * 1024**2:
            raise ValueError(f"{path.name}: missing, empty or over the upload size limit.")
        # Keep multipart headers single-line even for unusual uploaded names.
        name = path.name.replace('"', "_").replace("\r", "_").replace("\n", "_")
        payload.extend(("--" + boundary + '\r\nContent-Disposition: form-data; name="files"; filename="' +
                        name + '"\r\nContent-Type: application/octet-stream\r\n\r\n').encode())
        payload.extend(path.read_bytes())
        payload.extend(b"\r\n")
    payload.extend(("--" + boundary + "--\r\n").encode())
    try:
        status, _, body = api_request(
            "/api/v1/leads/extract", data=bytes(payload),
            content_type="multipart/form-data; boundary=" + boundary, timeout=120,
        )
    finally:
        payload.clear()
    assert status == 202
    job = json.loads(body)
    globals()["last_job_id"] = job["job_id"]
    return job

def wait_for_job(job_id, timeout=1200):
    deadline = time.monotonic() + timeout
    previous = None
    while True:
        job = api_json("/api/v1/leads/jobs/" + job_id)
        progress = (job["status"], job["processed"], job["successful"], job["failed"])
        if progress != previous:
            print(f"{job['status']}: {job['processed']}/{job['total']} cards, "
                  f"{job['successful']} successful, {job['failed']} failed")
            previous = progress
        if job["status"] in {"completed", "failed"}:
            display_results(job)
            return job
        if time.monotonic() >= deadline:
            raise TimeoutError("Batch still running. Call wait_for_job(last_job_id) to resume polling; do not resubmit.")
        time.sleep(2)

single_job = wait_for_job(submit_cards([SAMPLES / "standard.png"])["job_id"])
(REPORTS / "single-card-result.json").write_text(
    json.dumps(single_job, indent=2, ensure_ascii=False), encoding="utf-8"
)
truth = json.loads((SAMPLES / "expected.json").read_text())["standard.png"]
result = single_job["leads"][0] if single_job["leads"] else {}
if result.get("status") != "success":
    raise RuntimeError("The first real extraction failed. Inspect the result and backend log before continuing.")
exact = sum((result.get("lead") or {}).get(field) == truth[field] for field in FIELDS)
print(f"Exact matching fields: {exact}/7. Compare the table with the card; this is a diagnostic, not an accuracy guarantee.")
api_request("/api/v1/leads/jobs/" + single_job["job_id"], method="DELETE")


## 9. Run the real mixed-batch and Excel smoke check
The repository's smoke script sends three valid cards plus one corrupt file through the real HTTP API. It checks independent failure handling, edits an exported company and leading-zero phone, and verifies the XLSX structure. It also records exact-field diagnostics.

A successful smoke result establishes those backend checks only. Manually inspect the card values and downloaded workbook. Docker, Nginx, React and public access remain unverified.

In [ ]:
require_colab()
# Re-running this cell must not leave an old success report looking current.
for artifact in ("smoke-report.json", "smoke-report.xlsx"):
    (REPORTS / artifact).unlink(missing_ok=True)
run_command([PYTHON, BACKEND / "scripts/aws_smoke.py", "--colab-only",
             "--base-url", BASE_URL, "--samples", SAMPLES,
             "--output", REPORTS / "smoke-report.json"],
            env=backend_env, log_name="smoke-command.txt", timeout=1500)
print((REPORTS / "smoke-report.json").read_text())
colab_files.download(str(REPORTS / "smoke-report.xlsx"))


## 10. Optional: upload your cards, review/edit, and export
Run these cells only when you want to try your own images. JPG, JPEG, PNG and WEBP are accepted. Start with one or two cards.

Set `USE_MY_CARDS = True` to open a picker, or list already uploaded file paths. Review content before sharing this notebook: displayed results and uploaded files can contain contact information. The notebook writes uploads to its run directory and does not mount Google Drive.

In [ ]:
USE_MY_CARDS = False
CARD_PATHS = []  # Example: ["/content/card1.jpg", "/content/card2.png"]
my_job = None
reviewed_leads = []

if USE_MY_CARDS:
    require_colab()
    if not CARD_PATHS:
        chosen = colab_files.upload()
        uploaded_dir = RUN_ROOT / "uploaded-cards"
        uploaded_dir.mkdir(exist_ok=True)
        for index, (name, data) in enumerate(chosen.items()):
            leaf = Path(name).name
            path = uploaded_dir / f"{index:02d}-{leaf}"
            path.write_bytes(data)
            CARD_PATHS.append(str(path))
        del chosen
    my_job = wait_for_job(submit_cards(CARD_PATHS)["job_id"],
                          timeout=max(1200, len(CARD_PATHS) * 210))
    reviewed_leads = [
        dict(item["lead"]) for item in my_job["leads"]
        if item["status"] == "success" and item.get("lead")
    ]
    print("Review/edit the successful leads in the next cell before export.")
else:
    print("Skipped personal cards. Sections 8–9 already exercise the real backend.")


Edit the values below if needed, then run the export cell. Index `0` means the first successful card, not necessarily the first uploaded file. This exercises the same export API as the React frontend.

In [ ]:
# Uncomment and edit only the corrections you want:
# reviewed_leads[0]["company"] = "Correct company name"
# reviewed_leads[0]["phone"] = "+91 12345 67890"

if USE_MY_CARDS and reviewed_leads:
    print(json.dumps(reviewed_leads, indent=2, ensure_ascii=False))


In [ ]:
if USE_MY_CARDS:
    require_colab()
    if not reviewed_leads:
        raise RuntimeError("There are no successful leads to export. Review the extraction errors.")
    status, headers, workbook = api_request(
        "/api/v1/leads/export",
        data=json.dumps({"leads": reviewed_leads}).encode(),
        content_type="application/json", timeout=120,
    )
    assert status == 200 and workbook[:2] == b"PK", "Expected an XLSX response."
    MY_EXPORT = RUN_ROOT / "my-reviewed-leads.xlsx"
    MY_EXPORT.write_bytes(workbook)
    colab_files.download(str(MY_EXPORT))
    if my_job:
        api_request("/api/v1/leads/jobs/" + my_job["job_id"], method="DELETE")
else:
    print("Personal-card export skipped.")


## 11. Diagnostics and download the verification reports
Run this cell whenever startup or extraction fails (after section 7 has defined the helpers). It downloads a ZIP containing only the selected reports, synthetic results and backend logs, not your uploaded cards, personal Excel export, environment variables or model weights. Review logs before sharing.

If a request returns **429**, wait for the current batch; **404** means results expired or the backend restarted; **503** means the model is not ready. For out-of-memory errors, stop the backend, ensure no other GPU process is running, then retry a smaller card. If pip cannot resolve a pinned version, preserve the error and fix the dependency deliberately; do not drop every version pin.

In [ ]:
require_colab()
tail_backend()
print(subprocess.check_output(["nvidia-smi"], text=True))
report_zip = RUN_ROOT / "colab-backend-reports.zip"
allowed_reports = [
    "colab-patch-manifest.json", "pip-check.txt", "pip-freeze.txt", "gpu-check.txt",
    "venv-bootstrap.txt", "virtualenv-bootstrap.json", "virtualenv-repair.txt",
    "pytest.txt", "pytest.xml", "run-settings.json", "backend.log",
    "pip-freeze-error.txt", "package-versions.txt", "package-inventory.json",
    "single-card-result.json", "smoke-command.txt", "smoke-report.json", "smoke-report.xlsx",
]
with zipfile.ZipFile(report_zip, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in allowed_reports:
        artifact = REPORTS / name
        if artifact.is_file():
            bundle.write(artifact, arcname=name)
print("Saved reports:", report_zip)
colab_files.download(str(report_zip))


## 12. Stop the backend when finished
Run this cell to release GPU model memory. Model files remain cached on the Colab VM, so you can start again with section 7 while the same runtime exists. A deleted/reset runtime loses the environment, cache, cards and results: upload the ZIP again and rerun from section 1.

When you are done with the session, choose **Runtime → Disconnect and delete runtime**. Download anything you need first. This notebook does not keep the runtime alive automatically.

In [ ]:
require_colab()
if "stop_backend" in globals():
    stop_backend()
else:
    print("The backend has not been started by this notebook.")
